INSTALLAZIONE LIBRERIE

In [ ]:
pip install tsfel

In [ ]:
pip install imbalanced-learn

In [ ]:
pip install seaborn


In [ ]:
pip install matplotlib


IMPORT LIBRERIE

In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np
import random
import tsfel
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score
from imblearn.pipeline import Pipeline
from imblearn.combine import SMOTEENN
from sklearn.metrics import make_scorer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

Load Load input_matrix_train e input_matrix_test

In [ ]:
input_matrix_train = pd.read_csv('input_matrix_train.csv')

print("Matrice di train caricata:")
print(input_matrix_train)

input_matrix_test = pd.read_csv('input_matrix_test.csv')

print("Matrice di test caricata:")
print(input_matrix_test)




Preprocessing dati (scaling, clipping e smooting)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer




y = input_matrix_train["target"]
print("Valori unici di y (attività):", y.unique())
print("Occorrenze per ciascun attività:", y.value_counts())
print("Numero di righe e colonne di y:", y.shape)
groups = input_matrix_train["user"]
print("Valori unici di groups (user):", groups.unique())
print("Numero di righe e colonne di groups:", groups.shape)
X = input_matrix_train.drop(columns=['target', 'user'])
print("Numero di righe e colonne di X:", X.shape)
print("Prime 10 righe della matrice X")
print(X.head(10))


# Preparazione dei dati di test
y_test = input_matrix_test["target"]
X_test = input_matrix_test.drop(columns=["target", "timestamp"])  # Rimuovi colonne non necessarie
print("Classi uniche in y_test:", sorted(y_test.unique()))
print("Numero di classi in y_test:", len(y_test.unique()))





DecisionTree Model

In [ ]:
from imblearn.pipeline import Pipeline  # Usare il Pipeline di imblearn
from imblearn.combine import SMOTEENN  # SMOTEENN di imblearn
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import FunctionTransformer

# Creazione della pipeline
pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),       # Imputazione valori mancanti
    ('scaler', StandardScaler()),                      # Standardizzazione
    ('clipper', FunctionTransformer(lambda x: np.clip(x, -3, 3))),  # Clipping tra -3 e 3
    ('smoteenn', SMOTEENN()),                          # Bilanciamento con SMOTEENN
    ('classifier', DecisionTreeClassifier())           # Classificatore Decision Tree
])

# Definizione dei parametri per la GridSearch
parameters = {
    'classifier__criterion': ['gini', 'entropy'], 
    'classifier__max_depth': [3, 4, 5],
    'classifier__min_samples_split': [5, 10],
    'classifier__min_samples_leaf': [2]
}

# Creazione di GridSearchCV
gs = GridSearchCV(
    pipeline, 
    parameters, 
    cv=3, 
    scoring='f1_macro',  # Usa F1 macro per gestire classi sbilanciate
    verbose=50, 
    n_jobs=-1, 
    refit=True
)

# Addestramento con ricerca iperparametrica
gs.fit(X, y)

# Migliori parametri trovati
print("Best parameters found:", gs.best_params_)

# Valutazione sui dati di test
y_pred = gs.predict(X_test)

# Report di classificazione
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

# Matrice di confusione
conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.show()

SALVATAGGIO MIGLIOR MODELLO E MIGLIORI RISULTATI 

In [ ]:
import pickle
import json


In [ ]:
# Salvare miglior modello con pickle
#apri file best_rf_model.pkl in modalità scrittura binaria
with open("best_rf_model.pkl", "wb") as f:
    pickle.dump(best_rf_model, f) #primo parametro=oggetto da serializare, f=file aperto in modalità scrittura binaria dove verrà salvata la versione serializzata di best_rf_model

# Salva i migliori parametri e il punteggio in un file JSON
best_results = {
    "best_params": rf_cv.best_params_,
    "best_score": best_score
}
with open("best_rf_results.json", "w") as f: #apre un file in modalità scrittura
    json.dump(best_results, f) #Serializza (dump) il dizionario best_results in formato JSON e lo scrive nel file.
print("Modello e risultati salvati correttamente.")


In [ ]:
# Carica il modello
with open("best_rf_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

# Carica i risultati
with open("best_rf_results.json", "r") as f:
    loaded_results = json.load(f)

print("Modello e risultati caricati correttamente.")
print(f"Migliori parametri: {loaded_results['best_params']}")
print(f"Best F1 score: {loaded_results['best_score']:.2f}")


TRAINING FINALE SU TUTTO IL DATASET DI TRAIN CON IL MIGLIOR MODELLO E I MIGLIORI PARAMETRI 

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pickle
import json
RANDOM_STATE=18
# Carica i migliori parametri dal file JSON
with open("best_rf_results.json", "r") as f:
    loaded_results = json.load(f)

best_params = loaded_results["best_params"]  # Recupera i migliori parametri
print("Migliori parametri caricati:", best_params)

# Preparazione dei dati
y = input_matrix_train["target"]
X = input_matrix_train.drop(columns=["target", "user","timestamp"])  # Rimuovi le colonne non necessarie

# Crea la pipeline con SMOTE-ENN e RandomForestClassifier
steps = [
    ('smoteenn', SMOTEENN(random_state=RANDOM_STATE)),
    ('classifier', RandomForestClassifier(
        n_estimators=best_params["classifier__n_estimators"],
        max_features=best_params["classifier__max_features"],
        max_depth=best_params["classifier__max_depth"],
        criterion=best_params["classifier__criterion"],
        random_state=RANDOM_STATE
    ))
]
pipeline = Pipeline(steps=steps)

# Addestra il modello sui dati di training
print("Avvio del training sul dataset completo con SMOTE-ENN...")
pipeline.fit(X, y)
print("Pipeline addestrata con successo!")

# Salvataggio del modello finale
with open("final_rf_pipeline.pkl", "wb") as f:
    pickle.dump(pipeline, f)

print("Pipeline finale salvata in 'final_rf_pipeline.pkl'.")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

# Previsioni sui dati di addestramento
y_train_pred = pipeline.predict(X)

# Calcolare l'F1-score sui dati di addestramento
f1 = f1_score(y, y_train_pred, average='macro')
print(f"F1-score sui dati di addestramento: {f1:.2f}")

# Stampa il classification report
print("Classification Report sui dati di addestramento:")
print(classification_report(y, y_train_pred))

# Calcolare la matrice di confusione
conf_matrix = confusion_matrix(y, y_train_pred)

# Visualizzare la matrice di confusione
plt.figure(figsize=(15, 10))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=y.unique(), yticklabels=y.unique())
plt.title("Matrice di Confusione - Training Data")
plt.xlabel("Predizioni")
plt.ylabel("Valori Real")
plt.show()


PREPARAZIONE DATI DI TEST PER VALUTAZIONE FINALE

In [ ]:
#ho aggiunto lable 
for trace in datasetTracesTestFeatExtr:
    trace['TraceDataFeatExtr']['timestamp'] = [1 + i * 0.5 for i in range(len(trace['TraceDataFeatExtr']))]
    trace['TraceDataFeatExtr']['target']=trace['TraceIDFeatExtr'].split('_')[3] #creo una nuova colonna target a cui associo il valore di activity che sta nel traceid

    # Aggiorno la colonna 'target' per gestire l'attività '7' -> '07'
    trace['TraceDataFeatExtr']['target'] = trace['TraceDataFeatExtr']['target'].replace('7', '07')
    
print(datasetTracesTestFeatExtr[0]["TraceDataFeatExtr"])

#concatenato tutti i vari Dataframe =>ottengo un'unica grande matrice
input_matrix_test = pd.concat([item['TraceDataFeatExtr'] for item in datasetTracesTestFeatExtr], ignore_index=True)
print("dimensioni input_matrix:", input_matrix_test.shape)
print(input_matrix_test.head(10))

SALVATAGGIO INPUT MATRIX TEST

In [ ]:
# Salva la matrice in un file CSV
input_matrix_test.to_csv('input_matrix_test.csv', index=False)

DATI DI TEST, CONFUSION MATRIX E REPORT DI CLASSIFICAZIONE

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pickle

# Caricamento pipeline salvata
with open("final_rf_pipeline.pkl", "rb") as f:
    pipeline = pickle.load(f)
print("Pipeline caricata con successo.")

# Preparazione dei dati di test
y_test = input_matrix_test["target"]
X_test = input_matrix_test.drop(columns=["target", "timestamp"])  # Rimuovi colonne non necessarie
print("Classi uniche in y_test:", sorted(y_test.unique()))
print("Numero di classi in y_test:", len(y_test.unique()))



# uso del miglior modello per fare previsioni sui dati 
y_pred = pipeline.predict(X_test) #y_pred= array che contiene le etichette previste dal modello per ciascun esempio in X


# Calcolare l'F1-score sui dati di addestramento
f1 = f1_score(y_test, y_pred, average='macro')
print(f"F1-score sui dati di test: {f1:.2f}")

# Stampa il classification report
print("Classification Report sui dati di test:")
print(classification_report(y_test, y_pred))

# Calcolare la matrice di confusione
cm = confusion_matrix(y_test, y_pred)

# Visualizzare la matrice di confusione
plt.figure(figsize=(15, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=y_test.unique(), yticklabels=y_test.unique())
plt.title("Matrice di Confusione - Test Data")
plt.xlabel("Predizioni")
plt.ylabel("Valori Real")
plt.show()








In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

#report di classificazione
report_dict = classification_report(y_test, y_pred, output_dict=True)

#  precision, recall, F1-score per ciascuna classe
class_names = list(report_dict.keys())[:-3]  
precision = [report_dict[c]['precision'] for c in class_names]
recall = [report_dict[c]['recall'] for c in class_names]
f1_score = [report_dict[c]['f1-score'] for c in class_names]

# Creazione dei grafici a barre per Precision, Recall, e F1-score
x = np.arange(len(class_names))
width = 0.25

plt.figure(figsize=(12, 8))
plt.bar(x - width, precision, width, label='Precision', color='skyblue')
plt.bar(x, recall, width, label='Recall', color='lightgreen')
plt.bar(x + width, f1_score, width, label='F1-Score', color='salmon')

plt.xticks(x, class_names, rotation=45, fontsize=12)
plt.title('Precision, Recall, and F1-Score per Class', fontsize=16)
plt.xlabel('Class', fontsize=14)
plt.ylabel('Score', fontsize=14)
plt.legend(fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
